<a href="https://colab.research.google.com/github/raunakraj1310/SaginaBolo/blob/main/HindiNMTASR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dependencies

In [ ]:
# 1. Install dependencies
!pip -q install -U transformers torchaudio soundfile gradio huggingface_hub onnxruntime

print("✅ Dependencies installed")


In [ ]:
import os
import gradio as gr
import numpy as np
import torch
import librosa
import sys
import time
import json
import csv
import traceback
from pathlib import Path
from datetime import datetime, timezone
import transformers
import huggingface_hub

In [ ]:
os.environ["HF_HUB_DISABLE_XET"] = "1"
print("Transformers:", transformers.__version__)
print("HF Hub:", huggingface_hub.__version__)
print("Xet disabled:", os.environ["HF_HUB_DISABLE_XET"])

In [ ]:
# 0. Verify Colab GPU



print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "Enable a GPU in Colab: Runtime → Change runtime type → GPU"
    )

print("GPU:", torch.cuda.get_device_name(0))
print(
    f"VRAM: "
    f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB"
)

In [ ]:

from huggingface_hub import login
login()

# Transcription

In [ ]:
def transcribe(audio):

# Load an audio file
  wav, sr = torchaudio.load(audio)
  wav = torch.mean(wav, dim=0, keepdim=True)

  target_sample_rate = 16000  # Expected sample rate
  if sr != target_sample_rate:
      resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=target_sample_rate)
      wav = resampler(wav)


  # Perform ASR with RNNT decoding
  transcription_rnnt = model(wav, "hi", "rnnt")
  return(transcription_rnnt)


# IndicTrans2 HF Inference

We provide an example notebook on how to use our IndicTrans2 models which were originally trained with the fairseq to HuggingFace transformers for inference purpose.

## Setup

Please run the cells below to install the necessary dependencies.

In [ ]:
%%capture
!git clone https://github.com/AI4Bharat/IndicTrans2.git

In [ ]:
%%capture
%cd /content/IndicTrans2/huggingface_interface

In [ ]:
%%capture
!python3 -m pip install nltk sacremoses pandas regex mock transformers==4.53.2 mosestokenizer
!python3 -c "import nltk; nltk.download('punkt')"
!python3 -m pip install bitsandbytes scipy accelerate datasets
!python3 -m pip install sentencepiece

!git clone https://github.com/VarunGumma/IndicTransToolkit.git
%cd IndicTransToolkit
!python3 -m pip install --editable ./
%cd ..

**IMPORTANT : Restart your run-time first and then run the cells below.**

## ASR

In [ ]:
import os
import gradio as gr
import numpy as np
import torch
import librosa
import sys
import time
import json
import csv
import traceback
from pathlib import Path
from datetime import datetime, timezone
import transformers
import huggingface_hub

In [ ]:
from transformers import AutoModelForSeq2SeqLM, BitsAndBytesConfig, AutoTokenizer
from IndicTransToolkit.processor import IndicProcessor

BATCH_SIZE = 4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
quantization = None

In [ ]:
from transformers import AutoModel
import torch, torchaudio

# Load the model
model = AutoModel.from_pretrained("ai4bharat/indic-conformer-600m-multilingual", trust_remote_code=True)



In [ ]:
def transcribe(audio):

# Load an audio file
  wav, sr = torchaudio.load(audio)
  wav = torch.mean(wav, dim=0, keepdim=True)

  target_sample_rate = 16000  # Expected sample rate
  if sr != target_sample_rate:
      resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=target_sample_rate)
      wav = resampler(wav)


  # Perform ASR with RNNT decoding
  transcription_rnnt = model(wav, "hi", "rnnt")
  return(transcription_rnnt)


In [ ]:
def initialize_model_and_tokenizer(ckpt_dir, quantization):
    if quantization == "4-bit":
        qconfig = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
        )
    elif quantization == "8-bit":
        qconfig = BitsAndBytesConfig(
            load_in_8bit=True,
            bnb_8bit_use_double_quant=True,
            bnb_8bit_compute_dtype=torch.bfloat16,
        )
    else:
        qconfig = None

    tokenizer = AutoTokenizer.from_pretrained(ckpt_dir, trust_remote_code=True)
    model = AutoModelForSeq2SeqLM.from_pretrained(
        ckpt_dir,
        trust_remote_code=True,
        low_cpu_mem_usage=True,
        quantization_config=qconfig,
    )

    if qconfig == None:
        model = model.to(DEVICE)
        if DEVICE == "cuda":
            model.half()

    model.eval()

    return tokenizer, model


def batch_translate(input_sentences, src_lang, tgt_lang, model, tokenizer, ip):
    translations = []
    for i in range(0, len(input_sentences), BATCH_SIZE):
        batch = input_sentences[i : i + BATCH_SIZE]

        # Preprocess the batch and extract entity mappings
        batch = ip.preprocess_batch(batch, src_lang=src_lang, tgt_lang=tgt_lang)

        # Tokenize the batch and generate input encodings
        inputs = tokenizer(
            batch,
            truncation=True,
            padding="longest",
            return_tensors="pt",
            return_attention_mask=True,
        ).to(DEVICE)

        # Generate translations using the model
        with torch.no_grad():
            generated_tokens = model.generate(
                **inputs,
                use_cache=True,
                min_length=0,
                max_length=256,
                num_beams=5,
                num_return_sequences=1,
            )

        # Decode the generated tokens into text
        generated_tokens = tokenizer.batch_decode(
            generated_tokens,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True,
        )

        # Postprocess the translations, including entity replacement
        translations += ip.postprocess_batch(generated_tokens, lang=tgt_lang)

        del inputs
        torch.cuda.empty_cache()

    return translations

# Translation


In [ ]:
indic_en_ckpt_dir = "ai4bharat/indictrans2-indic-en-1B"  # ai4bharat/indictrans2-indic-en-dist-200M
indic_en_tokenizer, indic_en_model = initialize_model_and_tokenizer(indic_en_ckpt_dir, quantization)

ip = IndicProcessor(inference=True)


In [ ]:
def initialize_model_and_tokenizer(ckpt_dir, quantization):
    if quantization == "4-bit":
        qconfig = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
        )
    elif quantization == "8-bit":
        qconfig = BitsAndBytesConfig(
            load_in_8bit=True,
            bnb_8bit_use_double_quant=True,
            bnb_8bit_compute_dtype=torch.bfloat16,
        )
    else:
        qconfig = None

    tokenizer = AutoTokenizer.from_pretrained(ckpt_dir, trust_remote_code=True)
    model = AutoModelForSeq2SeqLM.from_pretrained(
        ckpt_dir,
        trust_remote_code=True,
        low_cpu_mem_usage=True,
        quantization_config=qconfig,
    )

    if qconfig == None:
        model = model.to(DEVICE)
        if DEVICE == "cuda":
            model.half()

    model.eval()

    return tokenizer, model


def batch_translate(input_sentences, src_lang, tgt_lang, model, tokenizer, ip):
    translations = []
    for i in range(0, len(input_sentences), BATCH_SIZE):
        batch = input_sentences[i : i + BATCH_SIZE]

        # Preprocess the batch and extract entity mappings
        batch = ip.preprocess_batch(batch, src_lang=src_lang, tgt_lang=tgt_lang)

        # Tokenize the batch and generate input encodings
        inputs = tokenizer(
            batch,
            truncation=True,
            padding="longest",
            return_tensors="pt",
            return_attention_mask=True,
        ).to(DEVICE)

        # Generate translations using the model
        with torch.no_grad():
            generated_tokens = model.generate(
                **inputs,
                use_cache=True,
                min_length=0,
                max_length=256,
                num_beams=5,
                num_return_sequences=1,
            )

        # Decode the generated tokens into text
        generated_tokens = tokenizer.batch_decode(
            generated_tokens,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True,
        )

        # Postprocess the translations, including entity replacement
        translations += ip.postprocess_batch(generated_tokens, lang=tgt_lang)

        del inputs
        torch.cuda.empty_cache()

    return translations

In [ ]:
def translate(transcript,src_lang="hin_Deva", tgt_lang="eng_Latn"):
  en_translations = batch_translate([transcript], src_lang, tgt_lang, indic_en_model, indic_en_tokenizer, ip)
  print(f"\n{src_lang} - {tgt_lang}")
  return (en_translations)
  # for input_sentence, translation in zip([transcript], en_translations):
  #     print(f"{src_lang}: {input_sentence}")
  #     print(f"{tgt_lang}: {translation}")

In [ ]:
text="""
उस्ताद राशिद ख़ान को एक संगीत-प्रेमी कैसे याद कर सकता है? इस पर ठहरता हूँ तो कुछ तस्वीरें ज़ेहन में आती हैं। ‘राग यमन’, ‘मारवा’, ‘सोहनी’, ‘मेघ’ और ‘ललित’ जैसे गंभीर ख़याल गाने वाले सिद्ध गायक—राशिद ख़ान। जहाँ भारतीय शास्त्रीय संगीत का बेहद सधा हुआ अनुशासन मौजूद है। जैसे एक समर्पित साधक अपने साधना में लीन हो। उप शास्त्रीय संगीत ‘ठुमरी’, पूर्वी, ‘टप्पे’ और ‘तराना’ इत्यादि गाने वाले— राशिद ख़ान। जहाँ एक आत्मीय खिलंदड़ापन और नोक-झोंक मौजूद हो। सभी संगतकारों को जगह देता हुआ, संवादी और लोकतांत्रिक— राशिद ख़ान। फ़िल्मी संगीत, जैज और कोक स्टूडियो जैसे युवाओं में लोकप्रिय गीत गाने वाले— राशिद ख़ान। एक परंपराबद्ध आधुनिक। जिसके पास कुछ नई और लोकप्रिय चीज़ों के लिए भी अवकाश है। यह अवकाश कुछ उस क़िस्म का है जहाँ नई पीढ़ी से एक ऐसा संवाद की कोई सूरत निकले। यह कुछ-कुछ सुनने की तैयारी कराने जैसा हो या एक कोशिश कि इस रास्ते कुछ नए श्रोताओं को जोड़ा जा सके। संकटमोचन मंदिर के परिसर में टहलते हुए अपने विचार में मग्न एक आस्तिक इंसान। अपने पुत्र को संकटमोचन मंदिर में संकटमोचन के सामने सजदा करने के लिए उत्साहित करता हुआ एक आत्मीय पिता। बनारस में गंगा घाट पर भीड़ से अपने को बचा कर टहलते हुए एक आम मनुष्य और संकटमोचन मंदिर में कार्यक्रम से पहले सुरमंडल मिलाते और कार्यक्रम के बाद अपने मुरीदों से मिलते हुए— राशिद ख़ान। ये वे छवियाँ हैं जिनसे मुझे राशिद ख़ान की याद आती है।
कला में छप्पन वर्ष की उम्र कोई बहुत बड़ी उम्र नहीं मानी जाती। यह कला के पकने की उम्र होती है। कला में निखार आने में एक लंबा वक़्त लगता है। यह राशिद ख़ान साहेब की ख़ुशनसीबी थी कि उन्हें कला जगत में वह पहचान और प्रसिद्धि दोनों मिली जिसके वह सबल हकदार थे। भारत रत्न पंडित भीमसेन जोशी ने राशिद ख़ान को ‘भारतीय शास्त्रीय संगीत का भविष्य’ कहा था।

‘आश्वस्ति’ कुछ अजीब शै है। मृत्यु अक्सर वह छोड़ जाती है, जिसमें हम जीवन के चिह्नों को तलाशते हैं। किंचित भारतीय श्रोताओं की यह बदनसीबी ही कही जाएगी कि अक्सर हमारे भविष्य को बहुत कम समय मिलता है। हम ऐसे त्रासद और हतभाग्य समय में जीने को विवश हैं, जहाँ जेनुइन प्रतिभाएँ उँगलियों पर गिनी जा सकती हैं। यह ऐसा समय है जहाँ कम मनुष्यों की भरमार है। बौनों और आत्ममुग्धों की संख्या हर क्षेत्र में बढती गई है। उस्ताद राशिद ख़ान के जाने से जो रिक्ति हुई है, इसकी भरपाई अब दूर तक संभव प्रतीत नहीं होती दिख रही है।
उस्ताद राशिद ख़ान रामपुर सहसवान घराने के ध्वजवाहक कलाकार। एक कलाकार की तैयारी और प्रस्तुति को अक्सर एक श्रोता समझ नहीं पाता। उसे एक उत्पाद के मानिंद एक तैयार चीज़ मिलती है। मेरा भी उस्ताद राशिद ख़ान से कुछ ऐसा ही संबंध रहा है—उत्पादक और भोक्ता का, किंतु संबंध चाहे जैसा हो एक समय के बाद उससे मोह होना स्वभाविक मानवीय गुण है। हमारी कल्पनाओं में भी यह रिक्ति कहीं नहीं थी। पिछले दिनों जब वह बीमार हुए तो यह उम्मीद थी कि उनके जीवन की यह मध्य-लय सम पर आकर उनकी ख़याल गायकी की ही तरह द्रुत में आ जाएगी, लेकिन यह ‘पुनः सवेरा एक और फेरा है जी का’ ही साबित हुआ। उन्होंने मध्य-लय के सम को ही अपना अंतिम सम माना।

"""
egtrans=translate(text,"hin_Deva", "eng_Latn")
egtrans

In [ ]:
print(egtrans)

In [ ]:
# src_lang, tgt_lang = "hin_Deva", "eng_Latn"
# transcript=transcribe("/content/OSR_in_000_0063_8k.wav")
# translate(transcript,"hin_Deva", "eng_Latn")

# Implementation

In [ ]:
def live(audio):
  transcripts=transcribe(audio)
  translate(transcripts,src_lang, tgt_lang)


# Record

In [ ]:
model.eval()

In [ ]:
import gradio as gr
import numpy as np
import torch
import torchaudio
from transformers import AutoModel


model.eval()

In [ ]:
model = model.to(DEVICE)
model.eval()

print("Model loaded.")
print("Device:", DEVICE)

In [ ]:
import gradio as gr
print("Gradio version:", gr.__version__)

In [ ]:
def transcribe(audio):
    if audio is None:
        return ""

    # Gradio may provide (sample_rate, numpy_array)
    sr, y = audio

    # Convert stereo -> mono
    if y.ndim > 1:
        y = y.mean(axis=1)

    # Convert to float32
    y = y.astype(np.float32)

    # Avoid division by zero
    if np.max(np.abs(y)) > 0:
        y = y / np.max(np.abs(y))

    # Convert numpy -> torch
    wav = torch.from_numpy(y).unsqueeze(0)

    # Resample to 16 kHz
    if sr != 16000:
        resampler = torchaudio.transforms.Resample(
            orig_freq=sr,
            new_freq=16000
        )
        wav = resampler(wav)

    # Move to GPU
    wav = wav.to(DEVICE)

    # Indic-Conformer RNNT decoding
    with torch.no_grad():
        transcription = model(wav, "hi", "rnnt")

    return transcription

# Successful

In [ ]:
import os
import base64
import torch
import torchaudio
import gradio as gr

from IPython.display import Javascript, display
from google.colab.output import eval_js


In [ ]:
# ---------------------------------------------------------
# 1. Browser microphone recorder
# ---------------------------------------------------------

def record_from_browser(seconds=5):
    js = Javascript("""
    async function recordAudio(seconds) {

        const stream = await navigator.mediaDevices.getUserMedia({
            audio: true
        });

        const recorder = new MediaRecorder(stream);
        const chunks = [];

        recorder.ondataavailable = event => {
            if (event.data.size > 0) {
                chunks.push(event.data);
            }
        };

        recorder.start();

        await new Promise(resolve =>
            setTimeout(resolve, seconds * 1000)
        );

        recorder.stop();

        await new Promise(resolve => {
            recorder.onstop = resolve;
        });

        stream.getTracks().forEach(track => track.stop());

        const blob = new Blob(chunks, {
            type: recorder.mimeType
        });

        const reader = new FileReader();

        return await new Promise(resolve => {
            reader.onloadend = () => resolve(reader.result);
            reader.readAsDataURL(blob);
        });
    }

    recordAudio
    """)

    display(js)

    data = eval_js(f"recordAudio({seconds})")

    audio_bytes = base64.b64decode(data.split(",")[1])

    webm_path = "/content/current_recording.webm"

    with open(webm_path, "wb") as f:
        f.write(audio_bytes)

    return webm_path



In [ ]:

# ---------------------------------------------------------
# 2. Convert browser recording to 16 kHz mono WAV
# ---------------------------------------------------------

def convert_to_wav(webm_path):

    wav_path = "/content/current_recording.wav"

    waveform, sr = torchaudio.load(webm_path)

    # Stereo -> mono
    if waveform.shape[0] > 1:
        waveform = torch.mean(waveform, dim=0, keepdim=True)

    # Resample to 16 kHz
    if sr != 16000:
        resampler = torchaudio.transforms.Resample(
            orig_freq=sr,
            new_freq=16000
        )
        waveform = resampler(waveform)

    torchaudio.save(
        wav_path,
        waveform,
        16000
    )

    return wav_path




In [ ]:
# ---------------------------------------------------------
# 3. Hindi ASR
# ---------------------------------------------------------

def transcribe_wav(wav_path):

    print("Processing:", wav_path)

    waveform, sr = torchaudio.load(wav_path)

    if waveform.shape[0] > 1:
        waveform = torch.mean(
            waveform,
            dim=0,
            keepdim=True
        )

    if sr != 16000:
        resampler = torchaudio.transforms.Resample(
            orig_freq=sr,
            new_freq=16000
        )
        waveform = resampler(waveform)

    print("Sample rate:", sr)
    print("Shape:", waveform.shape)

    result = model(
        waveform,
        "hi",
        "rnnt"
    )

    print("Transcription:", result)

    return str(result)



In [ ]:
def record_and_transcribe(seconds):

    print("\n🎤 Recording...")

    # -----------------------------------------
    # Record
    # -----------------------------------------

    webm_path = record_from_browser(seconds)

    print("Recorded:", webm_path)
    print("Size:", os.path.getsize(webm_path), "bytes")

    # -----------------------------------------
    # Convert to WAV
    # -----------------------------------------

    wav_path = convert_to_wav(webm_path)

    print("Converted:", wav_path)

    # -----------------------------------------
    # Process chunks
    # -----------------------------------------

    for hindi_text, english_text in transcribe_wav(wav_path):

        yield hindi_text, english_text, wav_path

In [ ]:
def transcribe_wav(wav_path, chunk_seconds=8):

    print("\n🎧 Loading:", wav_path)

    waveform, sr = torchaudio.load(wav_path)

    # Stereo → mono
    if waveform.shape[0] > 1:
        waveform = torch.mean(
            waveform,
            dim=0,
            keepdim=True
        )

    # Resample → 16 kHz
    if sr != 16000:
        resampler = torchaudio.transforms.Resample(
            orig_freq=sr,
            new_freq=16000
        )
        waveform = resampler(waveform)

    sr = 16000

    total_samples = waveform.shape[1]
    total_seconds = total_samples / sr

    print(f"🎵 Total duration: {total_seconds:.2f} seconds")

    chunk_samples = int(chunk_seconds * sr)

    hindi_chunks = []
    english_chunks = []

    for start in range(0, total_samples, chunk_samples):

        end = min(
            start + chunk_samples,
            total_samples
        )

        chunk = waveform[:, start:end]

        print(
            f"\n🧩 Processing "
            f"{start/sr:.1f}s → {end/sr:.1f}s"
        )

        if chunk.shape[1] < int(0.3 * sr):
            continue

        # ==========================================
        # HINDI ASR
        # ==========================================

        hindi_text = model(
            chunk,
            "hi",
            "rnnt"
        )

        hindi_text = str(hindi_text).strip()

        print("🇮🇳 Hindi:", hindi_text)

        # ==========================================
        # TRANSLATION
        # ==========================================

        english_text = ""

        if hindi_text:

            hindi_chunks.append(hindi_text)

            english_text = translate(
                hindi_text,
                src_lang="hin_Deva",
                tgt_lang="eng_Latn"
            )

            # batch_translate normally returns a list
            if isinstance(english_text, list):
                english_text = english_text[0]

            english_text = str(english_text).strip()

            english_chunks.append(english_text)

            print("🇬🇧 English:", english_text)

        # ==========================================
        # CURRENT CUMULATIVE RESULTS
        # ==========================================

        current_hindi = " ".join(hindi_chunks)
        current_english = " ".join(english_chunks)

        # Send update to Gradio
        yield current_hindi, current_english

    # ==============================================
    # FINAL RESULTS
    # ==============================================

    final_hindi = " ".join(hindi_chunks)
    final_english = " ".join(english_chunks)

    print("\n==============================")
    print("🇮🇳 FINAL HINDI:")
    print(final_hindi)

    print("\n🇬🇧 FINAL ENGLISH:")
    print(final_english)

    print("==============================")

    yield final_hindi, final_english

In [ ]:
with gr.Blocks() as demo:

    gr.Markdown(
        """
        # 🎤 Hindi Speech → English Translation

        ### Chunk-wise transcription and translation
        """
    )

    duration = gr.Slider(
        minimum=2,
        maximum=60,
        value=10,
        step=1,
        label="Recording duration (seconds)"
    )

    record_button = gr.Button(
        "🎙️ Record Hindi"
    )

    hindi_output = gr.Textbox(
        label="🇮🇳 Hindi Transcription",
        lines=8
    )

    english_output = gr.Textbox(
        label="🇬🇧 English Translation",
        lines=8
    )

    recorded_audio = gr.Audio(
        label="🎧 Recorded Audio",
        type="filepath"
    )

    record_button.click(
        fn=record_and_transcribe,
        inputs=duration,
        outputs=[
            hindi_output,
            english_output,
            recorded_audio
        ]
    )

demo.launch(debug=True)

# Model#2

In [ ]:
def pcm16_to_wav(pcm_bytes, sample_rate=16000):
    """
    Convert raw signed 16-bit PCM bytes to a WAV file.
    """
    wav_path = tempfile.NamedTemporaryFile(
        suffix=".wav",
        delete=False
    ).name

    pcm_tensor = torch.frombuffer(
        bytearray(pcm_bytes),
        dtype=torch.int16
    ).clone()

    # int16 -> float32
    waveform = pcm_tensor.float() / 32768.0

    # [samples] -> [1, samples]
    waveform = waveform.unsqueeze(0)

    torchaudio.save(
        wav_path,
        waveform,
        sample_rate
    )

    return wav_path

In [ ]:
def pcm16_to_wav(pcm_bytes, sample_rate=16000):
    """
    Convert raw signed 16-bit PCM bytes to a WAV file.
    """
    wav_path = tempfile.NamedTemporaryFile(
        suffix=".wav",
        delete=False
    ).name

    pcm_tensor = torch.frombuffer(
        bytearray(pcm_bytes),
        dtype=torch.int16
    ).clone()

    # int16 -> float32
    waveform = pcm_tensor.float() / 32768.0

    # [samples] -> [1, samples]
    waveform = waveform.unsqueeze(0)

    torchaudio.save(
        wav_path,
        waveform,
        sample_rate
    )

    return wav_path

In [ ]:
def translate_hi_to_en(hindi_text):
    if not hindi_text or not hindi_text.strip():
        return ""

    try:
        result = batch_translate([hindi_text])
        return result[0]
    except Exception as e:
        print("Translation error:", e)
        return ""

In [ ]:
def process_pcm_chunk(pcm_b64):
    print("\n==============================")
    print("NEW PCM CHUNK")
    print("==============================")

    try:
        # Base64 -> raw PCM bytes
        pcm_bytes = base64.b64decode(pcm_b64)

        print("PCM bytes:", len(pcm_bytes))

        # PCM -> valid WAV
        wav_path = pcm16_to_wav(
            pcm_bytes,
            sample_rate=16000
        )

        print("WAV:", wav_path)

        # ASR
        hindi = transcribe_wav(wav_path)

        print("Hindi:", hindi)

        # Translation
        english = translate_hi_to_en(hindi)

        print("English:", english)

        # Cleanup
        try:
            os.remove(wav_path)
        except:
            pass

        return hindi, english

    except Exception as e:
        print("\nERROR:", repr(e))

        return (
            f"ASR error: {e}",
            ""
        )

In [ ]:
def process_pcm_chunk(pcm_b64):
    print("\n==============================")
    print("NEW PCM CHUNK")
    print("==============================")

    try:
        # Base64 -> raw PCM bytes
        pcm_bytes = base64.b64decode(pcm_b64)

        print("PCM bytes:", len(pcm_bytes))

        # PCM -> valid WAV
        wav_path = pcm16_to_wav(
            pcm_bytes,
            sample_rate=16000
        )

        print("WAV:", wav_path)

        # ASR
        hindi = transcribe_wav(wav_path)

        print("Hindi:", hindi)

        # Translation
        english = translate_hi_to_en(hindi)

        print("English:", english)

        # Cleanup
        try:
            os.remove(wav_path)
        except:
            pass

        return hindi, english

    except Exception as e:
        print("\nERROR:", repr(e))

        return (
            f"ASR error: {e}",
            ""
        )

In [ ]:
PCM_HTML = """
<div style="
    padding:20px;
    border:1px solid #ccc;
    border-radius:12px;
    text-align:center;
">
    <button id="startBtn"
        style="font-size:18px;padding:12px 24px;">
        🎙️ Start Live Recording
    </button>

    <button id="stopBtn"
        style="font-size:18px;padding:12px 24px;margin-left:10px;"
        disabled>
        ⏹️ Stop
    </button>

    <div id="status"
        style="margin-top:15px;font-weight:bold;">
        Ready
    </div>
</div>
"""

In [ ]:
PCM_JS = r"""
async function() {

    const root = element;

    const startBtn = root.querySelector("#startBtn");
    const stopBtn = root.querySelector("#stopBtn");
    const status = root.querySelector("#status");

    let audioContext = null;
    let mediaStream = null;
    let source = null;
    let processor = null;

    let recording = false;

    // Browser-side PCM buffer
    let pcmChunks = [];
    let totalSamples = 0;

    // Send approximately every 5 seconds
    const chunkSeconds = 5;

    // We will resample browser audio to 16 kHz.
    const targetSampleRate = 16000;

    function mergeFloat32Arrays(chunks, totalLength) {

        const result = new Float32Array(totalLength);

        let offset = 0;

        for (const chunk of chunks) {
            result.set(chunk, offset);
            offset += chunk.length;
        }

        return result;
    }

    function floatToPCM16(float32Array) {

        const buffer = new ArrayBuffer(
            float32Array.length * 2
        );

        const view = new DataView(buffer);

        for (let i = 0; i < float32Array.length; i++) {

            let sample = Math.max(
                -1,
                Math.min(1, float32Array[i])
            );

            const value =
                sample < 0
                    ? sample * 32768
                    : sample * 32767;

            view.setInt16(
                i * 2,
                value,
                true
            );
        }

        return new Uint8Array(buffer);
    }

    function bytesToBase64(bytes) {

        let binary = "";

        const blockSize = 0x8000;

        for (
            let i = 0;
            i < bytes.length;
            i += blockSize
        ) {
            binary += String.fromCharCode(
                ...bytes.subarray(
                    i,
                    Math.min(i + blockSize, bytes.length)
                )
            );
        }

        return btoa(binary);
    }

    async function sendChunk() {

        if (totalSamples === 0) {
            return;
        }

        const merged = mergeFloat32Arrays(
            pcmChunks,
            totalSamples
        );

        const pcm16 = floatToPCM16(merged);

        const b64 = bytesToBase64(pcm16);

        console.log(
            "Sending PCM chunk:",
            merged.length,
            "samples"
        );

        // Gradio custom event
        await upload(
            new File(
                [pcm16],
                "chunk.pcm",
                {type: "application/octet-stream"}
            )
        );

        // Instead of relying on the uploaded file,
        // put the actual base64 PCM into props.value.
        props.value = b64;

        trigger("chunk");

        pcmChunks = [];
        totalSamples = 0;
    }

    function processAudio(event) {

        if (!recording) {
            return;
        }

        const input = event.inputBuffer.getChannelData(0);

        /*
         * Browser sample rate may be 44.1/48 kHz.
         *
         * Simple linear resampling to 16 kHz.
         */
        const ratio =
            targetSampleRate / audioContext.sampleRate;

        const outputLength =
            Math.floor(input.length * ratio);

        const output =
            new Float32Array(outputLength);

        for (let i = 0; i < outputLength; i++) {

            const position =
                i / ratio;

            const index =
                Math.floor(position);

            const fraction =
                position - index;

            if (index + 1 < input.length) {

                output[i] =
                    input[index] * (1 - fraction) +
                    input[index + 1] * fraction;

            } else {

                output[i] = input[index];
            }
        }

        pcmChunks.push(output);
        totalSamples += output.length;

        const requiredSamples =
            targetSampleRate * chunkSeconds;

        if (totalSamples >= requiredSamples) {
            sendChunk();
        }
    }

    startBtn.onclick = async () => {

        try {

            status.textContent =
                "Requesting microphone...";

            mediaStream =
                await navigator.mediaDevices.getUserMedia({
                    audio: {
                        channelCount: 1,
                        echoCancellation: false,
                        noiseSuppression: false,
                        autoGainControl: false
                    }
                });

            audioContext =
                new AudioContext();

            source =
                audioContext.createMediaStreamSource(
                    mediaStream
                );

            /*
             * ScriptProcessorNode is deprecated but is
             * widely supported and convenient for Colab.
             */
            processor =
                audioContext.createScriptProcessor(
                    4096,
                    1,
                    1
                );

            processor.onaudioprocess =
                processAudio;

            source.connect(processor);

            processor.connect(
                audioContext.destination
            );

            recording = true;

            pcmChunks = [];
            totalSamples = 0;

            startBtn.disabled = true;
            stopBtn.disabled = false;

            status.textContent =
                "🔴 Recording...";

        } catch (err) {

            console.error(err);

            status.textContent =
                "Microphone error: " + err;
        }
    };

    stopBtn.onclick = async () => {

        recording = false;

        // Send remaining audio
        if (totalSamples > 0) {
            await sendChunk();
        }

        if (processor) {
            processor.disconnect();
            processor = null;
        }

        if (source) {
            source.disconnect();
            source = null;
        }

        if (audioContext) {
            await audioContext.close();
            audioContext = null;
        }

        if (mediaStream) {

            mediaStream
                .getTracks()
                .forEach(track => track.stop());

            mediaStream = null;
        }

        startBtn.disabled = false;
        stopBtn.disabled = true;

        status.textContent =
            "Stopped";
    };
}
"""

In [ ]:
MIC_TEST_HTML = """
<div style="
    padding: 20px;
    border: 2px solid #ccc;
    border-radius: 12px;
    text-align: center;
">
    <button id="startBtn"
            style="font-size:18px; padding:12px 24px;">
        🎙️ Start Recording
    </button>

    <button id="stopBtn"
            style="font-size:18px; padding:12px 24px; margin-left:10px;"
            disabled>
        ⏹️ Stop
    </button>

    <div id="status"
         style="margin-top:15px; font-weight:bold;">
        Ready
    </div>
</div>
"""

In [ ]:
MIC_TEST_JS = r"""
async function() {

    const startBtn = element.querySelector("#startBtn");
    const stopBtn  = element.querySelector("#stopBtn");
    const status   = element.querySelector("#status");

    let stream = null;
    let audioContext = null;

    startBtn.onclick = async function() {

        console.log("START BUTTON CLICKED");

        try {

            status.textContent =
                "Requesting microphone permission...";

            stream = await navigator.mediaDevices.getUserMedia({
                audio: true
            });

            console.log("MICROPHONE ACCESS GRANTED");
            console.log(stream);

            audioContext = new AudioContext();

            console.log(
                "AudioContext sample rate:",
                audioContext.sampleRate
            );

            status.textContent =
                "🔴 Microphone is working!";

            startBtn.disabled = true;
            stopBtn.disabled = false;

        } catch (error) {

            console.error(
                "MICROPHONE ERROR:",
                error
            );

            status.textContent =
                "❌ Microphone error: " +
                error.name +
                " — " +
                error.message;
        }
    };


    stopBtn.onclick = async function() {

        console.log("STOP BUTTON CLICKED");

        if (stream) {

            stream.getTracks().forEach(
                track => track.stop()
            );

            stream = null;
        }

        if (audioContext) {

            await audioContext.close();
            audioContext = null;
        }

        status.textContent =
            "Stopped";

        startBtn.disabled = false;
        stopBtn.disabled = true;
    };
}
"""

In [ ]:
import gradio as gr

with gr.Blocks() as demo:

    gr.Markdown("# JavaScript Test")

    gr.HTML("""
    <button id="testBtn"
            style="font-size:20px;padding:15px;">
        CLICK ME
    </button>

    <div id="result"
         style="margin-top:20px;font-size:20px;">
        Waiting...
    </div>

    <script>
        document.getElementById("testBtn").onclick = function() {
            document.getElementById("result").innerText =
                "✅ JavaScript is working!";
        };
    </script>
    """)

demo.launch(debug=True)

In [ ]:
with gr.Blocks() as demo:

    gr.Markdown("# 🎙️ Microphone Test")

    mic = gr.HTML(
        html_template=MIC_TEST_HTML,
        js_on_load=MIC_TEST_JS
    )

demo.launch(
    debug=True
)

In [ ]:
with gr.Blocks() as demo:

    gr.Markdown(
        """
        # 🎙️ Hindi → English Live Speech

        Speak Hindi. Audio is captured as PCM,
        converted to WAV, transcribed, and translated
        chunk-by-chunk.
        """
    )

    mic = gr.HTML(
        html_template=PCM_HTML,
        js_on_load=PCM_JS
    )

    gr.Markdown("## 🇮🇳 Hindi Transcription")

    hindi_output = gr.Textbox(
        label="Hindi",
        lines=8,
        interactive=False
    )

    gr.Markdown("## 🇬🇧 English Translation")

    english_output = gr.Textbox(
        label="English",
        lines=8,
        interactive=False
    )

    mic.chunk(
        fn=process_pcm_chunk,
        inputs=mic,
        outputs=[
            hindi_output,
            english_output
        ]
    )


demo.launch(
    debug=True
)

# Test #3

In [ ]:
from IPython.display import Javascript, display
from google.colab.output import eval_js
from base64 import b64decode
import time

def record_pcm_from_browser(seconds=5):

    js = Javascript("""
    async function recordPCM(seconds) {

        const stream = await navigator.mediaDevices.getUserMedia({
            audio: {
                channelCount: 1,
                echoCancellation: false,
                noiseSuppression: false,
                autoGainControl: false
            }
        });

        const audioContext = new AudioContext();

        await audioContext.resume();

        const source =
            audioContext.createMediaStreamSource(stream);

        const processor =
            audioContext.createScriptProcessor(
                4096,
                1,
                1
            );

        const chunks = [];

        processor.onaudioprocess = function(event) {

            const input =
                event.inputBuffer.getChannelData(0);

            chunks.push(new Float32Array(input));
        };

        source.connect(processor);
        processor.connect(audioContext.destination);

        await new Promise(resolve => {
            setTimeout(resolve, seconds * 1000);
        });

        processor.disconnect();
        source.disconnect();

        stream.getTracks().forEach(
            track => track.stop()
        );

        await audioContext.close();

        /*
         * Merge Float32 PCM chunks
         */
        let totalLength = 0;

        for (const chunk of chunks) {
            totalLength += chunk.length;
        }

        const merged =
            new Float32Array(totalLength);

        let offset = 0;

        for (const chunk of chunks) {
            merged.set(chunk, offset);
            offset += chunk.length;
        }

        /*
         * Convert Float32 -> Int16 PCM
         */
        const pcm16 =
            new Int16Array(merged.length);

        for (let i = 0; i < merged.length; i++) {

            let sample =
                Math.max(
                    -1,
                    Math.min(1, merged[i])
                );

            pcm16[i] =
                sample < 0
                    ? sample * 32768
                    : sample * 32767;
        }

        /*
         * Return:
         * sample rate + PCM bytes
         */
        const bytes =
            new Uint8Array(pcm16.buffer);

        let binary = "";

        const blockSize = 0x8000;

        for (
            let i = 0;
            i < bytes.length;
            i += blockSize
        ) {
            binary += String.fromCharCode(
                ...bytes.subarray(
                    i,
                    Math.min(
                        i + blockSize,
                        bytes.length
                    )
                )
            );
        }

        return JSON.stringify({
            sampleRate: audioContext.sampleRate,
            pcm: btoa(binary)
        });
    }

    recordPCM
    """)

    display(js)

    result = eval_js(
        f"recordPCM({seconds})"
    )

    import json

    result = json.loads(result)

    pcm_bytes = b64decode(result["pcm"])

    sample_rate = result["sampleRate"]

    return pcm_bytes, sample_rate

In [ ]:
pcm_bytes, browser_sr = record_pcm_from_browser(5)

print("Recording successful")
print("PCM bytes:", len(pcm_bytes))
print("Browser sample rate:", browser_sr)

In [ ]:
import torch
import torchaudio

def pcm_bytes_to_wav(
    pcm_bytes,
    input_sample_rate,
    output_path="/content/live_chunk.wav"
):

    pcm = torch.frombuffer(
        bytearray(pcm_bytes),
        dtype=torch.int16
    ).clone()

    waveform = pcm.float() / 32768.0

    waveform = waveform.unsqueeze(0)

    if input_sample_rate != 16000:

        resampler = torchaudio.transforms.Resample(
            orig_freq=input_sample_rate,
            new_freq=16000
        )

        waveform = resampler(waveform)

    torchaudio.save(
        output_path,
        waveform,
        16000
    )

    return output_path

In [ ]:
wav_path = pcm_bytes_to_wav(
    pcm_bytes,
    browser_sr
)

print("WAV created:", wav_path)

waveform, sr = torchaudio.load(wav_path)

print("Shape:", waveform.shape)
print("Sample rate:", sr)
print("Duration:", waveform.shape[1] / sr, "seconds")

In [ ]:
hindi_text = transcribe_wav(wav_path)

print("Hindi:")
print(hindi_text)

In [ ]:
def transcribe_wav(wav_path):
    waveform, sr = torchaudio.load(wav_path)

    # Convert to mono
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)

    # Resample to 16 kHz
    if sr != 16000:
        resampler = torchaudio.transforms.Resample(
            orig_freq=sr,
            new_freq=16000
        )
        waveform = resampler(waveform)

    with torch.inference_mode():
        result = model(
            waveform,
            "hi",
            "rnnt"
        )

    # The model returns a generator
    if hasattr(result, "__iter__") and not isinstance(result, str):
        result = list(result)

    return result

In [ ]:
hindi_text = transcribe_wav(wav_path)

print("Hindi:")
print(hindi_text)
print(type(hindi_text))

print(repr(hindi_text))

In [ ]:
def process_one_chunk(seconds=5):

    print("\n" + "=" * 50)
    print(f"🎙️ Recording {seconds} seconds...")
    print("=" * 50)

    # 1. Browser microphone → PCM
    pcm_bytes, browser_sr = record_pcm_from_browser(seconds)

    print("PCM bytes:", len(pcm_bytes))
    print("Browser sample rate:", browser_sr)

    # 2. PCM → WAV
    wav_path = pcm_bytes_to_wav(
        pcm_bytes,
        browser_sr,
        "/content/live_chunk.wav"
    )

    # 3. WAV → Hindi ASR
    hindi = transcribe_wav(wav_path)

    print("\n🇮🇳 Hindi:")
    print(hindi)

    # 4. Hindi → English
    english = batch_translate(
        [hindi],
        "hin_Deva",
        "eng_Latn",
        indic_en_model,
        indic_en_tokenizer,
        ip
    )[0]

    print("\n🇬🇧 English:")
    print(english)

    return hindi, english

In [ ]:
hindi, english = process_one_chunk(30)

In [ ]:
def live_chunks(chunk_seconds=5, max_chunks=None):

    chunk_number = 0

    while True:

        chunk_number += 1

        print("\n")
        print("=" * 60)
        print(f"🎙️ CHUNK {chunk_number}")
        print("=" * 60)

        # Record
        pcm_bytes, browser_sr = record_pcm_from_browser(
            chunk_seconds
        )

        print(
            f"PCM: {len(pcm_bytes)} bytes | "
            f"SR: {browser_sr}"
        )

        # PCM → WAV
        wav_path = f"/content/live_chunk_{chunk_number}.wav"

        pcm_bytes_to_wav(
            pcm_bytes,
            browser_sr,
            wav_path
        )

        # ASR
        hindi = transcribe_wav(wav_path)

        print("\n🇮🇳 Hindi:")
        print(hindi)

        # Translation
        if hindi.strip():

            english = batch_translate(
                [hindi],
                "hin_Deva",
                "eng_Latn",
                indic_en_model,
                indic_en_tokenizer,
                ip
            )[0]

        else:
            english = ""

        print("\n🇬🇧 English:")
        print(english)

        # Remove WAV
        try:
            os.remove(wav_path)
        except:
            pass

        # Return current result
        yield chunk_number, hindi, english

        # Optional testing limit
        if (
            max_chunks is not None
            and chunk_number >= max_chunks
        ):
            break

In [ ]:
for chunk_no, hindi, english in live_chunks(
    chunk_seconds=5,
    max_chunks=3
):

    print(
        f"\n✅ Finished chunk {chunk_no}"
    )

## Continous

In [ ]:
from IPython.display import Javascript, display
from google.colab.output import eval_js
from base64 import b64decode
import json
import torch
import torchaudio
import os

In [ ]:
def record_continuous_pcm(
    total_seconds=30,
    window_seconds=10,
    step_seconds=5
):

    js = Javascript(f"""
    async function recordContinuous() {{

        const stream =
            await navigator.mediaDevices.getUserMedia({{
                audio: {{
                    channelCount: 1,
                    echoCancellation: false,
                    noiseSuppression: false,
                    autoGainControl: false
                }}
            }});

        const audioContext = new AudioContext();

        await audioContext.resume();

        const source =
            audioContext.createMediaStreamSource(stream);

        const processor =
            audioContext.createScriptProcessor(
                4096,
                1,
                1
            );

        const chunks = [];

        processor.onaudioprocess = function(event) {{

            const input =
                event.inputBuffer.getChannelData(0);

            chunks.push(
                new Float32Array(input)
            );
        }};

        source.connect(processor);
        processor.connect(
            audioContext.destination
        );

        console.log(
            "Continuous recording started"
        );

        await new Promise(resolve => {{
            setTimeout(
                resolve,
                {total_seconds * 1000}
            );
        }});

        processor.disconnect();
        source.disconnect();

        stream.getTracks().forEach(
            track => track.stop()
        );

        const sampleRate =
            audioContext.sampleRate;

        await audioContext.close();

        /*
         * Merge all browser audio.
         */

        let totalLength = 0;

        for (const chunk of chunks) {{
            totalLength += chunk.length;
        }}

        const merged =
            new Float32Array(totalLength);

        let offset = 0;

        for (const chunk of chunks) {{
            merged.set(chunk, offset);
            offset += chunk.length;
        }}

        /*
         * Convert Float32 → Int16 PCM.
         */

        const pcm16 =
            new Int16Array(merged.length);

        for (let i = 0; i < merged.length; i++) {{

            let sample =
                Math.max(
                    -1,
                    Math.min(1, merged[i])
                );

            pcm16[i] =
                sample < 0
                    ? sample * 32768
                    : sample * 32767;
        }}

        const bytes =
            new Uint8Array(pcm16.buffer);

        let binary = "";

        const blockSize = 0x8000;

        for (
            let i = 0;
            i < bytes.length;
            i += blockSize
        ) {{

            binary += String.fromCharCode(
                ...bytes.subarray(
                    i,
                    Math.min(
                        i + blockSize,
                        bytes.length
                    )
                )
            );
        }}

        return JSON.stringify({{
            sampleRate: sampleRate,
            pcm: btoa(binary)
        }});
    }}

    recordContinuous
    """)

    display(js)

    result = eval_js(
        "recordContinuous()"
    )

    result = json.loads(result)

    pcm_bytes = b64decode(
        result["pcm"]
    )

    sample_rate = result["sampleRate"]

    return pcm_bytes, sample_rate

In [ ]:
def split_pcm_into_windows(
    pcm_bytes,
    sample_rate,
    window_seconds=10,
    step_seconds=5
):

    pcm = torch.frombuffer(
        bytearray(pcm_bytes),
        dtype=torch.int16
    ).clone()

    total_samples = len(pcm)

    window_samples = int(
        window_seconds * sample_rate
    )

    step_samples = int(
        step_seconds * sample_rate
    )

    windows = []

    start = 0

    while start + window_samples <= total_samples:

        end = start + window_samples

        window = pcm[start:end]

        windows.append(window)

        start += step_samples

    return windows

In [ ]:
def split_pcm_into_windows(
    pcm_bytes,
    sample_rate,
    window_seconds=10,
    step_seconds=5
):

    pcm = torch.frombuffer(
        bytearray(pcm_bytes),
        dtype=torch.int16
    ).clone()

    total_samples = len(pcm)

    window_samples = int(
        window_seconds * sample_rate
    )

    step_samples = int(
        step_seconds * sample_rate
    )

    windows = []

    start = 0

    while start + window_samples <= total_samples:

        end = start + window_samples

        window = pcm[start:end]

        windows.append(window)

        start += step_samples

    return windows

In [ ]:
def split_pcm_into_windows(
    pcm_bytes,
    sample_rate,
    window_seconds=10,
    step_seconds=5
):

    pcm = torch.frombuffer(
        bytearray(pcm_bytes),
        dtype=torch.int16
    ).clone()

    total_samples = len(pcm)

    window_samples = int(
        window_seconds * sample_rate
    )

    step_samples = int(
        step_seconds * sample_rate
    )

    windows = []

    start = 0

    while start + window_samples <= total_samples:

        end = start + window_samples

        window = pcm[start:end]

        windows.append(window)

        start += step_samples

    return windows

In [ ]:
windows = split_pcm_into_windows(
    pcm_bytes,
    browser_sr,
    window_seconds=10,
    step_seconds=5
)

print(
    "Number of windows:",
    len(windows)
)

for i, window in enumerate(windows, 1):

    print(
        f"Window {i}: "
        f"{len(window)} samples"
    )

In [ ]:
results = []

for i, window in enumerate(
    windows,
    start=1
):

    print("\n")
    print("=" * 60)
    print(
        f"PROCESSING WINDOW {i}"
    )
    print("=" * 60)

    hindi, english = process_pcm_window(
        window,
        browser_sr,
        i
    )

    results.append({
        "window": i,
        "start": (i - 1) * 5,
        "end": (i - 1) * 5 + 10,
        "hindi": hindi,
        "english": english
    })

In [ ]:
print("Total PCM bytes:", len(pcm_bytes))
print("Browser sample rate:", browser_sr)

print("\nWindows:", len(windows))

for r in results:
    print("\n" + "=" * 60)
    print(f"Window {r['window']}: {r['start']}–{r['end']} sec")
    print("Hindi:", r["hindi"])
    print("English:", r["english"])

In [ ]:
pcm_bytes, browser_sr = record_continuous_pcm(
    total_seconds=30,
    window_seconds=10,
    step_seconds=5
)

print("Recording complete")
print("PCM bytes:", len(pcm_bytes))
print("Browser sample rate:", browser_sr)

In [ ]:
windows = split_pcm_into_windows(
    pcm_bytes,
    browser_sr,
    window_seconds=10,
    step_seconds=5
)

print("Number of windows:", len(windows))

for i, window in enumerate(windows, 1):
    print(
        f"Window {i}: "
        f"{len(window)} samples"
    )

In [ ]:
def process_pcm_window(
    pcm_tensor,
    sample_rate,
    window_number
):

    wav_path = (
        f"/content/sliding_window_"
        f"{window_number}.wav"
    )

    # PCM int16 → float waveform
    waveform = (
        pcm_tensor.float() / 32768.0
    )

    waveform = waveform.unsqueeze(0)

    # Resample browser audio → 16 kHz
    if sample_rate != 16000:

        resampler = torchaudio.transforms.Resample(
            orig_freq=sample_rate,
            new_freq=16000
        )

        waveform = resampler(waveform)

    # Save temporary WAV
    torchaudio.save(
        wav_path,
        waveform,
        16000
    )

    # Hindi ASR
    hindi = transcribe_wav(wav_path)

    print(f"\n🇮🇳 Window {window_number} Hindi:")
    print(hindi)

    # Hindi → English
    if hindi.strip():

        english = batch_translate(
            [hindi],
            "hin_Deva",
            "eng_Latn",
            indic_en_model,
            indic_en_tokenizer,
            ip
        )[0]

    else:
        english = ""

    print(f"\n🇬🇧 Window {window_number} English:")
    print(english)

    # Delete temporary WAV
    try:
        os.remove(wav_path)
    except:
        pass

    return hindi, english

In [ ]:
results = []

for i, window in enumerate(windows, start=1):

    print("\n" + "=" * 60)
    print(f"PROCESSING WINDOW {i}")
    print("=" * 60)

    hindi, english = process_pcm_window(
        window,
        browser_sr,
        i
    )

    results.append({
        "window": i,
        "start": (i - 1) * 5,
        "end": (i - 1) * 5 + 10,
        "hindi": hindi,
        "english": english
    })

In [ ]:
def split_pcm_into_windows(
    pcm_bytes,
    sample_rate,
    window_seconds=10,
    step_seconds=5
):

    pcm = torch.frombuffer(
        bytearray(pcm_bytes),
        dtype=torch.int16
    ).clone()

    total_samples = len(pcm)

    window_samples = int(
        window_seconds * sample_rate
    )

    step_samples = int(
        step_seconds * sample_rate
    )

    windows = []

    start = 0

    while start + window_samples <= total_samples:

        end = start + window_samples

        windows.append(
            pcm[start:end]
        )

        start += step_samples

    return windows

In [ ]:
from IPython.display import Javascript, display
from google.colab.output import eval_js
from base64 import b64decode
import json
import time


def record_live_sliding(
    total_seconds=30,
    window_seconds=10,
    step_seconds=5
):

    js = Javascript(f"""
    async function liveSliding() {{

        const stream =
            await navigator.mediaDevices.getUserMedia({{
                audio: {{
                    channelCount: 1,
                    echoCancellation: false,
                    noiseSuppression: false,
                    autoGainControl: false
                }}
            }});

        const audioContext = new AudioContext();

        await audioContext.resume();

        const source =
            audioContext.createMediaStreamSource(stream);

        const processor =
            audioContext.createScriptProcessor(
                4096,
                1,
                1
            );

        const chunks = [];

        processor.onaudioprocess = function(event) {{

            const input =
                event.inputBuffer.getChannelData(0);

            chunks.push(
                new Float32Array(input)
            );
        }};

        source.connect(processor);
        processor.connect(audioContext.destination);

        const sampleRate =
            audioContext.sampleRate;

        const startTime =
            performance.now();

        // Keep microphone alive continuously
        await new Promise(resolve => {{
            setTimeout(
                resolve,
                {total_seconds * 1000}
            );
        }});

        processor.disconnect();
        source.disconnect();

        stream.getTracks().forEach(
            track => track.stop()
        );

        await audioContext.close();

        // Merge all audio
        let totalLength = 0;

        for (const chunk of chunks) {{
            totalLength += chunk.length;
        }}

        const merged =
            new Float32Array(totalLength);

        let offset = 0;

        for (const chunk of chunks) {{
            merged.set(chunk, offset);
            offset += chunk.length;
        }}

        // Float32 → Int16 PCM
        const pcm16 =
            new Int16Array(merged.length);

        for (
            let i = 0;
            i < merged.length;
            i++
        ) {{

            let sample =
                Math.max(
                    -1,
                    Math.min(1, merged[i])
                );

            pcm16[i] =
                sample < 0
                    ? sample * 32768
                    : sample * 32767;
        }}

        const bytes =
            new Uint8Array(pcm16.buffer);

        let binary = "";

        const blockSize = 0x8000;

        for (
            let i = 0;
            i < bytes.length;
            i += blockSize
        ) {{

            binary += String.fromCharCode(
                ...bytes.subarray(
                    i,
                    Math.min(
                        i + blockSize,
                        bytes.length
                    )
                )
            );
        }}

        return JSON.stringify({{
            sampleRate: sampleRate,
            pcm: btoa(binary)
        }});
    }}

    liveSliding
    """)

    display(js)

    result = eval_js(
        "liveSliding()"
    )

    result = json.loads(result)

    pcm_bytes = b64decode(
        result["pcm"]
    )

    sample_rate = result["sampleRate"]

    return pcm_bytes, sample_rate

In [ ]:
pcm_bytes, browser_sr = record_live_sliding(
    total_seconds=30,
    window_seconds=10,
    step_seconds=5
)

print("Recording complete")
print("PCM bytes:", len(pcm_bytes))
print("Browser sample rate:", browser_sr)

In [ ]:
windows = split_pcm_into_windows(
    pcm_bytes,
    browser_sr,
    window_seconds=10,
    step_seconds=5
)

print("Number of windows:", len(windows))

for i, window in enumerate(windows, 1):
    print(
        f"Window {i}: "
        f"{len(window)} samples"
    )

In [ ]:
results = []

for i, window in enumerate(
    windows,
    start=1
):

    print("\n" + "=" * 60)
    print(f"PROCESSING WINDOW {i}")
    print("=" * 60)

    hindi, english = process_pcm_window(
        window,
        browser_sr,
        i
    )

    results.append({
        "window": i,
        "start": (i - 1) * 5,
        "end": (i - 1) * 5 + 10,
        "hindi": hindi,
        "english": english
    })

In [ ]:
def process_pcm_window(
    pcm_tensor,
    sample_rate,
    window_number
):

    waveform = (
        pcm_tensor.float() / 32768.0
    ).unsqueeze(0)

    # Resample to 16 kHz
    if sample_rate != 16000:
        resampler = torchaudio.transforms.Resample(
            orig_freq=sample_rate,
            new_freq=16000
        )
        waveform = resampler(waveform)

    # ASR directly from waveform
    with torch.inference_mode():
        hindi = model(
            waveform,
            "hi",
            "rnnt"
        )

    if hasattr(hindi, "__iter__") and not isinstance(hindi, str):
        hindi = list(hindi)

    if isinstance(hindi, list):
        hindi = hindi[0] if hindi else ""

    hindi = str(hindi).strip()

    print(f"\n🇮🇳 Window {window_number}:")
    print(hindi)

    # Translation
    if hindi:

        english = batch_translate(
            [hindi],
            "hin_Deva",
            "eng_Latn",
            indic_en_model,
            indic_en_tokenizer,
            ip
        )[0]

    else:
        english = ""

    print(f"\n🇬🇧 Window {window_number}:")
    print(english)

    return hindi, english

In [ ]:
hindi, english = process_pcm_window(
    windows[0],
    browser_sr,
    1
)

# Live Implementation

In [ ]:
from IPython.display import Javascript, display
from google.colab.output import eval_js
from base64 import b64decode
import json


def live_record_windows(
    total_seconds=30,
    window_seconds=10,
    step_seconds=5
):

    js = Javascript(f"""
    async function liveRecord() {{

        const stream =
            await navigator.mediaDevices.getUserMedia({{
                audio: {{
                    channelCount: 1,
                    echoCancellation: false,
                    noiseSuppression: false,
                    autoGainControl: false
                }}
            }});

        const audioContext = new AudioContext();

        await audioContext.resume();

        const source =
            audioContext.createMediaStreamSource(stream);

        const processor =
            audioContext.createScriptProcessor(
                4096,
                1,
                1
            );

        const chunks = [];

        processor.onaudioprocess = function(event) {{

            const input =
                event.inputBuffer.getChannelData(0);

            chunks.push(
                new Float32Array(input)
            );
        }};

        source.connect(processor);
        processor.connect(audioContext.destination);

        const sampleRate =
            audioContext.sampleRate;

        const start =
            performance.now();

        // Keep microphone open continuously
        await new Promise(resolve => {{
            setTimeout(
                resolve,
                {total_seconds * 1000}
            );
        }});

        processor.disconnect();
        source.disconnect();

        stream.getTracks().forEach(
            track => track.stop()
        );

        await audioContext.close();

        // Merge PCM
        let totalLength = 0;

        for (const chunk of chunks) {{
            totalLength += chunk.length;
        }}

        const merged =
            new Float32Array(totalLength);

        let offset = 0;

        for (const chunk of chunks) {{
            merged.set(chunk, offset);
            offset += chunk.length;
        }}

        // Float32 → Int16
        const pcm16 =
            new Int16Array(merged.length);

        for (
            let i = 0;
            i < merged.length;
            i++
        ) {{

            let sample =
                Math.max(
                    -1,
                    Math.min(1, merged[i])
                );

            pcm16[i] =
                sample < 0
                    ? sample * 32768
                    : sample * 32767;
        }}

        const bytes =
            new Uint8Array(pcm16.buffer);

        let binary = "";

        const blockSize = 0x8000;

        for (
            let i = 0;
            i < bytes.length;
            i += blockSize
        ) {{

            binary += String.fromCharCode(
                ...bytes.subarray(
                    i,
                    Math.min(
                        i + blockSize,
                        bytes.length
                    )
                )
            );
        }}

        return JSON.stringify({{
            sampleRate: sampleRate,
            pcm: btoa(binary)
        }});
    }}

    liveRecord
    """)

    display(js)

    result = eval_js("liveRecord()")

    result = json.loads(result)

    pcm_bytes = b64decode(
        result["pcm"]
    )

    return (
        pcm_bytes,
        result["sampleRate"]
    )

In [ ]:
print("🎙️ Starting microphone...")
print("Speak continuously for 30 seconds...")

pcm_bytes, browser_sr = live_record_windows(
    total_seconds=30,
    window_seconds=10,
    step_seconds=5
)

print("\n✅ Recording finished")
print("PCM bytes:", len(pcm_bytes))
print("Sample rate:", browser_sr)

In [ ]:
windows = split_pcm_into_windows(
    pcm_bytes,
    browser_sr,
    window_seconds=10,
    step_seconds=5
)

print("Windows:", len(windows))

for i, window in enumerate(windows, 1):
    print(
        f"Window {i}: "
        f"{len(window)} samples"
    )

In [ ]:
live_results = []

for i, window in enumerate(
    windows,
    start=1
):

    print("\n" + "=" * 70)
    print(f"🔴 LIVE WINDOW {i}")
    print("=" * 70)

    hindi, english = process_pcm_window(
        window,
        browser_sr,
        i
    )

    live_results.append({
        "window": i,
        "start": (i - 1) * 5,
        "end": (i - 1) * 5 + 10,
        "hindi": hindi,
        "english": english
    })

## b

In [ ]:
from IPython.display import Javascript, display
from google.colab.output import eval_js
from base64 import b64decode
import json
import torch
import torchaudio


def record_pcm_chunk(seconds=5):

    js = Javascript(f"""
    async function recordChunk() {{

        const stream =
            await navigator.mediaDevices.getUserMedia({{
                audio: {{
                    channelCount: 1,
                    echoCancellation: false,
                    noiseSuppression: false,
                    autoGainControl: false
                }}
            }});

        const audioContext =
            new AudioContext();

        await audioContext.resume();

        const source =
            audioContext.createMediaStreamSource(
                stream
            );

        const processor =
            audioContext.createScriptProcessor(
                4096,
                1,
                1
            );

        const chunks = [];

        processor.onaudioprocess =
            function(event) {{

                const input =
                    event.inputBuffer
                        .getChannelData(0);

                chunks.push(
                    new Float32Array(input)
                );
            }};

        source.connect(processor);
        processor.connect(
            audioContext.destination
        );

        await new Promise(resolve => {{
            setTimeout(
                resolve,
                {seconds * 1000}
            );
        }});

        processor.disconnect();
        source.disconnect();

        stream.getTracks().forEach(
            track => track.stop()
        );

        const sampleRate =
            audioContext.sampleRate;

        await audioContext.close();

        let totalLength = 0;

        for (const chunk of chunks) {{
            totalLength += chunk.length;
        }}

        const merged =
            new Float32Array(totalLength);

        let offset = 0;

        for (const chunk of chunks) {{
            merged.set(chunk, offset);
            offset += chunk.length;
        }}

        const pcm16 =
            new Int16Array(
                merged.length
            );

        for (
            let i = 0;
            i < merged.length;
            i++
        ) {{

            let sample =
                Math.max(
                    -1,
                    Math.min(
                        1,
                        merged[i]
                    )
                );

            pcm16[i] =
                sample < 0
                    ? sample * 32768
                    : sample * 32767;
        }}

        const bytes =
            new Uint8Array(
                pcm16.buffer
            );

        let binary = "";

        const blockSize = 0x8000;

        for (
            let i = 0;
            i < bytes.length;
            i += blockSize
        ) {{

            binary += String.fromCharCode(
                ...bytes.subarray(
                    i,
                    Math.min(
                        i + blockSize,
                        bytes.length
                    )
                )
            );
        }}

        return JSON.stringify({{
            sampleRate: sampleRate,
            pcm: btoa(binary)
        }});
    }}

    recordChunk
    """)

    display(js)

    result = eval_js("recordChunk()")

    result = json.loads(result)

    return (
        b64decode(result["pcm"]),
        result["sampleRate"]
    )

In [ ]:
def append_pcm(
    buffer,
    new_pcm_bytes
):

    new_pcm = torch.frombuffer(
        bytearray(new_pcm_bytes),
        dtype=torch.int16
    ).clone()

    if buffer is None:
        return new_pcm

    return torch.cat([
        buffer,
        new_pcm
    ])

In [ ]:
rolling_buffer = None

sample_rate = None

chunk_seconds = 5
window_seconds = 10
total_seconds = 30

live_results = []

print("🎙️ LIVE TRANSLATION STARTING")
print("Speak continuously...")
print()

for step in range(
    total_seconds // chunk_seconds
):

    print(
        f"\n🎤 Recording chunk "
        f"{step + 1}..."
    )

    pcm_chunk, sr = record_pcm_chunk(
        chunk_seconds
    )

    sample_rate = sr

    rolling_buffer = append_pcm(
        rolling_buffer,
        pcm_chunk
    )

    # Need at least 10 seconds
    required_samples = int(
        window_seconds * sample_rate
    )

    if len(rolling_buffer) < required_samples:

        print(
            "⏳ Building audio buffer..."
        )

        continue

    # Keep latest 10 seconds
    window = rolling_buffer[
        -required_samples:
    ]

    print(
        "\n" + "=" * 70
    )

    print(
        f"🔴 LIVE WINDOW "
        f"{step + 1}"
    )

    print(
        "=" * 70
    )

    hindi, english = process_pcm_window(
        window,
        sample_rate,
        step + 1
    )

    live_results.append({
        "window": step + 1,
        "hindi": hindi,
        "english": english
    })

    # Prevent unlimited memory growth
    rolling_buffer = rolling_buffer[
        -required_samples:
    ]
